In [ ]:
os.environ["PYOPENGL_PLATFORM"] = "egl" #opengl seems to only work with TPU
!PYOPENGL_PLATFORM=egl python -c "from OpenGL import EGL"
print(os.environ['PYOPENGL_PLATFORM']) 

In [ ]:
def batch_method_for_keys(batch, method, keys):
    for key in keys:
        
        # Make a batch, apply method and copy

        pure_batch = [value[key] for value in batch]
        result = method(pure_batch)
        for i in len(batch):
            batch[i][key]=result



In [ ]:
media = EncodedVideo.from_path("/home/jeanmanuel/memory-perceiver/data/hSN3RZh3iL8.mp4")
video = (media.get_clip(start_sec=0, end_sec=0.2))["video"]
print(video.shape)
out = body_from_video(video)
vis_vid = rearrange(video, 'c t h w -> t c h w')
# vis_vid = out['img']
frames, _, h, w = vis_vid.shape
rendered_video=[]
to_render=[]
bbox = out['bbox']
mesh = out['smplx_mesh_cam']
cam_params = batch_intrinsic_camera(bbox, w, h)

# With translation
# cam_params = batch_intrinsic_camera_original(frames, w ,h)
# mesh = mesh + bbox_trans(mesh, bbox, out['cam_param'], device)

rendered_video = render_silhouettes(vis_vid, mesh, repeat(smpl_x.face, 'd1 d2 -> f d1 d2', f = frames), cam_params, device, hd=False)
# cv2.imwrite('render_original_img' + str(i) + '.jpg', rendered_img)
    
ani=ts.show_video(rendered_video)
HTML(ani.to_jshtml())
vis_vid = rearrange(video, 'c t h w -> t c h w')
frames, _, h, w = vis_vid.shape
rendered_video=[]
to_render=[]
for i in range(frames):
    bbox = out['bbox'][i]
    mesh = out['smplx_mesh_cam'][i].detach().cpu().numpy()
    vis_img = rearrange(vis_vid[i], 'c h w -> h w c').cpu().numpy().copy()
    cam_params = intrinsic_camera(bbox, w, h)
    rendered_img = render_mesh(vis_img, mesh, smpl_x.face, cam_params)
    to_render.append([vis_img, mesh, smpl_x.face, cam_params])
    rendered_video.append(torch.tensor(rendered_img))
    # cv2.imwrite('render_original_img' + str(i) + '.jpg', rendered_img)
    
rendered_video = torch.stack(rendered_video)
ani=ts.show_video(rendered_video)
HTML(ani.to_jshtml())
joint_proj, joint_cam, mesh_cam = model_smplx.module.get_coord(out['smplx_root_pose'], out['smplx_body_pose'], out['smplx_lhand_pose'],
                         out['smplx_rhand_pose'], out['smplx_jaw_pose'], out['smplx_shape'], 
                         out['smplx_expr'], out['cam_trans'], 'test') 
for custom file, change path
est_sources = model_sepformer.separate_file(path='speechbrain/sepformer-wham-enhancement/example_wham.wav') 

torchaudio.save("enhanced_wham.wav", est_sources[:, :, 0].detach().cpu(), 8000)
with torch.no_grad():
    te=model_sepformer(torch.randn(2,1000))
rearrange(te, 'b l () -> b l')
t1 = torchaudio.load("enhanced_wham.wav")

In [ ]:
# from thinkFuture import ThinkFuture

# Simple inference

# learning_rate = 0.001
# num_epochs = 1
# optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
# train_loss = []

# for epoch in tqdm(range(num_epochs)):

#     model = model.train()

#     for i, (inputs, targets) in enumerate(trainloader):

#         # transform inputs

#         inputs = [transform_audio(nb_frames)(transform_video(nb_frames)(vid)) for vid in inputs]
#         targets  = torch.stack([transform_audio(frames_future)(vid)["audio"] for vid in targets])
        
#         # vectorize

#         batch = input_to_tensor(inputs, model.video_latent, model.audio_latent)

#         # zero the parameter gradients

#         optimizer.zero_grad()

#         # forward

#         x, aux_loss = model(batch, queries = audio_query)
#         project = projection(x)
#         x = rearrange(project ,  "b t t1 -> b (t t1)")
#         loss = aux_loss + F.l1_loss(x, targets)
        
#         # update model params
#         loss.backward()
#         optimizer.step()
#         model.reset_memories()

#         print(loss.detach().item())
#         train_loss.append(loss.detach().item())

#     plt.plot(train_loss)
#     plt.show()

In [ ]:
# def bbox_trans(mesh, bbox, cam_params, device):
#     bbox = bbox.to(device)
#     focal = cam_params['focal'][0].to(device)
#     princpt = cam_params['princpt'][0].to(device)
#     Z = torch.mean(mesh[:, :, 2], dim=-1).to(device)
#     X = torch.max(mesh[:, :, 0], dim=-1).values.to(device)
#     Y = torch.max(mesh[:, :, 1], dim=-1).values.to(device)
#     trans_x = 2 * ( (bbox[:, 2]+bbox[:, 0])*w/2 - princpt[0]) * X / ( (bbox[:, 2]-bbox[:, 0])*w )
#     trans_y = 2 * ( (bbox[:, 3]+bbox[:, 1])*h/2 - princpt[1]) * Y  / ( (bbox[:, 3]-bbox[:, 1])*h )
#     trans_z = -Z + focal[0] * trans_x / ( (bbox[:, 2]+bbox[:, 0])*w/2 - princpt[0])
#     trans = rearrange([trans_x, trans_y, trans_z], 'xyz f -> f xyz')
#     _, d, _ = out['smplx_mesh_cam'].shape
#     return repeat(trans, 'f xyz -> f d xyz', d=d)

In [ ]:
import concurrent
from more_itertools import grouper

# Slice inputs

to_slice = [(paths[i], media_starts[i]+start, media_starts[i]+start+duration) for i in range(len_batch)]
# executor = concurrent.futures.ProcessPoolExecutor(num_workers)
# futures = [executor.submit(my_multiprocesses.slice_medias, group) for group in grouper(num_workers, to_slice) ]
# concurrent.futures.wait(futures)
# inputs = sum([future.result() for future in futures], [])    
# executor.shutdown()  
inputs = my_multiprocesses.slice_medias(to_slice)

# Slice targets

to_slice = [(paths[i], media_starts[i]+start+duration, media_starts[i]+start+duration+duration_future) for i in range(len_batch)]
# executor = concurrent.futures.ProcessPoolExecutor(num_workers)
# futures = [executor.submit(my_multiprocesses.slice_medias, group) for group in grouper(num_workers, to_slice) ]
# concurrent.futures.wait(futures)
# targets = sum([future.result() for future in futures], [])  
# executor.shutdown()  
targets = my_multiprocesses.slice_medias(to_slice)

In [ ]:
# transform inputs and targets

# if( start < duration ):
#     for i in range(len(inputs)):
#         inputs[i].update(body_from_video(UniformTemporalSubsample(nb_frames)(inputs[i]['video'])))
# inputs = [transform_audio(nb_frames)(transform_video(nb_frames)(sequence)) for sequence in inputs]
# targets_audio  = torch.stack([transform_audio(frames_future)(target)["audio"] for target in targets]).to(device)
# targets_body  = batch_dictionary([body_from_video(UniformTemporalSubsample(frames_future)(target["video"])) for target in targets])
# targets_body = {part: item.to(device) for part, item in targets_body.items()}
# if(acting):
#     with torch.no_grad():
#         targets_audio=upsampler(model_sepformer(subsampler(targets_audio.to(deviceEval))).squeeze(-1)).to(device)

# print('transform inputs and extract body: ', time.time()-t0)
# t0 = time.time()

In [ ]:
# install pytorch3d
# import sys
# import torch
# pyt_version_str=torch.__version__.split("+")[0].replace(".", "")
# version_str="".join([
#     f"py3{sys.version_info.minor}_cu",
#     torch.version.cuda.replace(".",""),
#     f"_pyt{pyt_version_str}"
# ])
# !pip install --no-index --no-cache-dir pytorch3d -f https://dl.fbaipublicfiles.com/pytorch3d/packaging/wheels/{version_str}/download.html

In [ ]:
    def reset_memories(self, indexes=None):

        # Reset memory for new lifelong learning

        device =  self.cross_mem.device
        max_batch_size, mem_size, queries_dim = self.cross_mem.shape
        _, cmem_size, _ = self.cross_cmem.shape
        if indexes is None:
            indexes=torch.arange(max_batch_size)
        self.cross_mem[indexes] = torch.zeros(len(indexes), mem_size, queries_dim, device=device)
        self.cross_cmem[indexes] = torch.zeros(len(indexes), cmem_size, queries_dim, device=device)
        for i in range(self.depth):
            if ( i % 2 == 1):
                self.layers_mem[i//2][indexes] = torch.zeros(len(indexes), mem_size, queries_dim, device=device)
                self.layers_cmem[i//2][indexes] = torch.zeros(len(indexes), cmem_size, queries_dim, device=device)